# XGBoost SHAP

## Import libraries

In [1]:
# Turn warnings off to keep notebook tidy
import warnings
warnings.filterwarnings("ignore")

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import shap

from dataclasses import dataclass
from sklearn.metrics import auc
from sklearn.metrics import roc_curve
from xgboost import XGBClassifier

## General settings

In [2]:
# K folds in data
n_kfold = 5
# Disability thresholds for binary classification models
list_binary_thresholds = [0, 1, 2, 3, 4, 5]
n_binary_models = len(list_binary_thresholds)
# Surrogate time for no thrombolysis (for SHAP dependence plots)
surrogate_time_for_no_thrombolysis = 9999

File paths

In [3]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''
    image_save_path: str = './saved_images'
    model_save_path: str = './saved_models'
    data_save_path: str = './saved_data'
    data_read_path: str = './data_processing/output/kfold_5fold'
    model_text: str = (f'xgb_{n_kfold}fold_binary_model')
    notebook: str = '02_' # Prefix to file name

paths = Paths()

Create output folders if needed

In [4]:
path = paths.image_save_path
if not os.path.exists(path):
    os.makedirs(path)
        
path = paths.model_save_path
if not os.path.exists(path):
    os.makedirs(path)

path = paths.data_save_path
if not os.path.exists(path):
    os.makedirs(path)

## Features to use

In [5]:
# Features to use
selected_features = ["prior_disability", "stroke_severity", "stroke_team", 
                     "age", "onset_to_thrombolysis_time", "any_afib_diagnosis", 
                     "precise_onset_known"]

# Feature name dictionary for plotting
dict_feature_names = {}
dict_feature_names["prior_disability"] = "Prior disability (mRS)"
dict_feature_names["stroke_severity"] = "Stroke severity (NIHSS)"
dict_feature_names["stroke_team"] = "Stroke team"
dict_feature_names["age"] = "Age (years)"
dict_feature_names["onset_to_thrombolysis_time"] = "Onset to thrombolysis time (minutes)"
dict_feature_names["any_afib_diagnosis"] = "Atrial fibrilation diagnosis"
dict_feature_names["precise_onset_known"] = "Precise onset time known"
dict_feature_names['discharge_disability'] = "Discharge disabiliity (mRS)"

# Number of features
n_features = len(selected_features)

# Target feature
target_feature = 'discharge_disability'

## Load data

In [6]:
X_train_kfold, y_train_kfold, X_test_kfold, y_test_kfold = [], [], [], []

for k in range(n_kfold):
    # Read in training set, restrict to chosen features & store
    filename = os.path.join(paths.data_read_path, 
                            ('train_' + str(k) + '.csv'))
    train = pd.read_csv(filename)
    X_train = train[selected_features]
    y_train = train[target_feature]
    X_train_kfold.append(X_train)
    y_train_kfold.append(y_train)

    # Read in test set, restrict to chosen features & store
    filename = os.path.join(paths.data_read_path, 
                            ('test_' + str(k) + '.csv'))
    test = pd.read_csv(filename)
    X_test = test[selected_features]
    y_test = test[target_feature]
    X_test_kfold.append(X_test)
    y_test_kfold.append(y_test)

## One-hot encode categorical features

In [7]:
def convert_feature_to_one_hot(df, feature_name, prefix):
    """
    Converts a categorical feature into a one hot encoded feature
    
    Args:
        df [dataframe]: training or test dataset
        feature_name [str]: feature to convert to one hot encoding
        prefix [str]: string to use on new feature

    Return:
        df [dataframe]: One hot encoded representation of the feature
    """

    # One hot encode a feature
    df_feature = pd.get_dummies(
        df[feature_name], prefix=prefix)
    df = pd.concat([df, df_feature], axis=1)
    df.drop(feature_name, axis=1, inplace=True)

    return(df)

In [8]:
def convert_feature_to_one_hot(df, feature_name, prefix):
    """
    Converts a categorical feature into a one hot encoded feature
    
    Args:
        df [dataframe]: training or test dataset
        feature_name [str]: feature to convert to one hot encoding
        prefix [str]: string to use on new feature

    Return:
        df [dataframe]: One hot encoded representation of the feature
    """

    # One hot encode a feature
    df_feature = pd.get_dummies(
        df[feature_name], prefix=prefix)
    df = pd.concat([df, df_feature], axis=1)
    df.drop(feature_name, axis=1, inplace=True)

    return(df)

Keep copy of original data, with 'Stroke team' not one-hot encoded

In [9]:
train_stroke_team_kfold, test_stroke_team_kfold = [], []
for k in range(n_kfold):
    train_stroke_team_kfold.append(X_train_kfold[k]["stroke_team"].values)
    test_stroke_team_kfold.append(X_test_kfold[k]["stroke_team"].values)

For both train and test dataset, convert "stroke_team" feature to one hot encoded.

In [10]:
feature = "stroke_team"
prefix = "team"
for k in range(n_kfold):
    if feature in list(X_train):
        X_train_kfold[k] = convert_feature_to_one_hot(X_train_kfold[k], feature, prefix)
        X_test_kfold[k] = convert_feature_to_one_hot(X_test_kfold[k], feature, prefix)

Feature names with one hot encoding (using data for kfold0)

In [11]:
feature_names_ohe = list(X_train_kfold[0])
n_features_ohe = len(feature_names_ohe)

In [12]:
print(f"There are {len(selected_features)} original features "
      f"(before one-hot encoding)")
print(f"There are {n_features_ohe} features (after one-hot encoding)")

There are 7 original features (before one-hot encoding)
There are 124 features (after one-hot encoding)


## Create data frames for each disability threshold

In [13]:
def create_df_of_binary_target_features(target_feature_data, binary_thresholds):
    """ 
    Given a feature, return a dataframe of all of the different binary versions 
    depending on the threshold used

    Args:
        data [dataframe]: patient feature values
        target_feature [string]: feature name of target feature
        binary_thresholds [list]: list of thresholds to use for each version of 
                    a binary target feature

    Return:
        df [dataframe]: row per instance, column per threshold. Containing 
        feature values for patients with binary representation of the target 
        feature, based on the threshold value 
    
    """    
    df = pd.DataFrame()

    for threshold in binary_thresholds:
        binary_target_feature_name = f'{target_feature}_bin_{threshold}'

        # For the new feature, set all to 0    
        df[binary_target_feature_name] = 0

        # Then set to 1 those patients that are the threshold value or less
        df[binary_target_feature_name] = (
                                    (target_feature_data <= threshold) * 1)
        
    return(df)

Create dataframe of the binary target features, store in a list per kfold.

In [14]:
list_df_all_y_bin_train_kfold, list_df_all_y_bin_test_kfold = [], []

for k in range(n_kfold):
    list_df_all_y_bin_train_kfold.append(create_df_of_binary_target_features(
                                    y_train_kfold[k], list_binary_thresholds))
    list_df_all_y_bin_test_kfold.append(create_df_of_binary_target_features(
                                    y_test_kfold[k], list_binary_thresholds))

## Load models

In [15]:
models=dict()
for binary_threshold in list_binary_thresholds:
    for k in range(n_kfold):
        # Load pickled model
        filename = os.path.join(paths.model_save_path, 
                                    (paths.notebook + paths.model_text +  "_mrs" +
                                    str(binary_threshold) + '_kfold' + str(k) + '.p'))
        with open(filename, 'rb') as file:
            models[(binary_threshold, k)] = pickle.load(file)

### Test loaded models

In [16]:
def calculate_predicted_probabilities(model, X_data):
    """ 
    Given a model and input data, return the models probability and prediction
    for each instance.
    
    Args:
        model [xgboost classifier object]: trained model
        X_data [dataframe]: input features for model

    Return:
        y_probs [array]: the probability of being in each target feature class
        y_pred [array]: the prediction (the class with the largest probability)
    """

    # Get and store predicted probabilities
    y_probs = model.predict_proba(X_data)

    # Get and store predicted class
    y_pred = model.predict(X_data)

    return(y_probs, y_pred)

In [17]:
# Store all the results for the binary model (for the 5 kfolds)
dict_binary_model_results = {}

# Include the new binary feature we calculate (replaces the multiclass feature)
for binary_threshold in list_binary_thresholds:
    # Set up the target feature name for this threshold
    binary_target_feature_name = f'{target_feature}_bin_{binary_threshold}'

    # Initialise lists
    y_pred_binary_kfold = []
    y_probs_binary_kfold = []
    shap_values_extended_kfold = []
    model_kfold = []
    model_dict = dict()
    accuracy_kfold = []
    feature_importance_kfold = []
    rocauc_kfold = []
    y_bin_test_kfold = []
    y_bin_train_kfold = []
    y_error_kfold = []

    for k in range(n_kfold):

        # Get target feature for this threshold
        y_bin_train_kfold.append(list_df_all_y_bin_train_kfold[k][binary_target_feature_name])

        ## Select XGBoost model
        model_kfold.append(models[(binary_threshold, k)])

        # Store predictions for test dataset
        (y_probs, y_pred) = calculate_predicted_probabilities(model_kfold[k], X_test_kfold[k])
        y_pred_binary_kfold.append(y_pred)
        y_probs_binary_kfold.append(y_probs[:,1])
       
        # Get ROC AUC
        y_bin_test_kfold.append(list_df_all_y_bin_test_kfold[k][binary_target_feature_name])
        fpr, tpr, thresholds = roc_curve(y_bin_test_kfold[k], y_probs[:,1])
        rocauc_kfold.append(auc(fpr, tpr))

        # Calculate error
        y_error_kfold.append(y_bin_test_kfold[k] - y_pred)
        accuracy_kfold.append(np.mean(y_error_kfold[k]==0))

        # Get and store feature importances
        feature_importance_kfold.append(model_kfold[k].feature_importances_)

    # Print accuracy summary
    accuracy_mean = np.mean(accuracy_kfold)
    rocauc_mean =  np.mean(rocauc_kfold)
    threshold_index = list_binary_thresholds.index(binary_threshold)

    print(f"Model using binary threshold {binary_threshold}:")
    print (f'Accuracy: {accuracy_mean:0.3f} '
           f'(std across {n_kfold} kfolds: {np.std(accuracy_kfold):0.3f})')

    print (f'ROC AUC: {rocauc_mean:0.3f} '
           f'(std across {n_kfold} kfolds: {np.std(rocauc_kfold):0.3f})')
 
    print()

    # Save kfold results in dictionary for this threshold
    dict_binary_model_results[f"accuracy_kfold_list_mrs{binary_threshold}"] = accuracy_kfold
    dict_binary_model_results[f"rocauc_kfold_list_mrs{binary_threshold}"] = rocauc_kfold
    dict_binary_model_results[f"y_pred_kfold_list_mrs{binary_threshold}"] = y_pred_binary_kfold
    dict_binary_model_results[f"y_probs_kfold_list_mrs{binary_threshold}"] = y_probs_binary_kfold
    dict_binary_model_results[f"shap_values_extended_kfold_list_mrs{binary_threshold}"] = shap_values_extended_kfold
    dict_binary_model_results[f"model_kfold_list_mrs{binary_threshold}"] = model_kfold
    dict_binary_model_results[f"feature_importance_kfold_list_mrs{binary_threshold}"] = feature_importance_kfold
    dict_binary_model_results[f"y_bin_test_kfold_list_mrs{binary_threshold}"] = y_bin_test_kfold 
    dict_binary_model_results[f"y_bin_train_kfold_list_mrs{binary_threshold}"] = y_bin_train_kfold
    dict_binary_model_results[f"y_error_kfold_list_mrs{binary_threshold}"] = y_error_kfold

Model using binary threshold 0:
Accuracy: 0.882 (std across 5 kfolds: 0.001)
ROC AUC: 0.853 (std across 5 kfolds: 0.002)

Model using binary threshold 1:
Accuracy: 0.775 (std across 5 kfolds: 0.001)
ROC AUC: 0.852 (std across 5 kfolds: 0.002)

Model using binary threshold 2:
Accuracy: 0.800 (std across 5 kfolds: 0.001)
ROC AUC: 0.876 (std across 5 kfolds: 0.001)

Model using binary threshold 3:
Accuracy: 0.843 (std across 5 kfolds: 0.001)
ROC AUC: 0.890 (std across 5 kfolds: 0.002)

Model using binary threshold 4:
Accuracy: 0.882 (std across 5 kfolds: 0.001)
ROC AUC: 0.893 (std across 5 kfolds: 0.001)

Model using binary threshold 5:
Accuracy: 0.896 (std across 5 kfolds: 0.001)
ROC AUC: 0.868 (std across 5 kfolds: 0.002)



## Fit SHAP models

In [18]:
for binary_threshold in list_binary_thresholds:
    for k in range(n_kfold):
        print(f"Calculating SHAP values for binary threshold {binary_threshold} and kfold {k}")
        model = models[(binary_threshold, k)]
        shap_values_extended = shap.TreeExplainer(model).shap_values(X_test_kfold[k])[1]

Calculating SHAP values for binary threshold 0 and kfold 0
Calculating SHAP values for binary threshold 0 and kfold 1
Calculating SHAP values for binary threshold 0 and kfold 2
Calculating SHAP values for binary threshold 0 and kfold 3
Calculating SHAP values for binary threshold 0 and kfold 4
Calculating SHAP values for binary threshold 1 and kfold 0
Calculating SHAP values for binary threshold 1 and kfold 1
Calculating SHAP values for binary threshold 1 and kfold 2
Calculating SHAP values for binary threshold 1 and kfold 3
Calculating SHAP values for binary threshold 1 and kfold 4
Calculating SHAP values for binary threshold 2 and kfold 0
Calculating SHAP values for binary threshold 2 and kfold 1
Calculating SHAP values for binary threshold 2 and kfold 2
Calculating SHAP values for binary threshold 2 and kfold 3
Calculating SHAP values for binary threshold 2 and kfold 4
Calculating SHAP values for binary threshold 3 and kfold 0
Calculating SHAP values for binary threshold 3 and kfold